# Show-o2: text-derived red steering

**Can a text-derived red steering signal increase red elements in generated images?**

Select a GPU runtime (A100 / High RAM when available). Run setup once, restart
the session, then load the model. Push this branch before using it in Colab.

Implemented: baseline generation and text-only direction extraction.
Image-position injection is the next step. Real Show-o2 GPU execution remains
unverified. The general demo stays in `colab.ipynb`.


In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)


## 1. Setup

Run once in a fresh Colab runtime. Use our experiment branch and a fixed upstream commit.


In [ ]:
import subprocess

branch = "experiment/text-red-steering"
commit = "45a5a2de01d1ebd10cd5864d29310a76476cdf23"

subprocess.run(["git", "clone", "--branch", branch,
                "https://github.com/Nicerova7/show-o2-space.git",
                "/content/show-o2-space"], check=True)
subprocess.run(["git", "clone", "https://github.com/showlab/Show-o.git",
                "/content/Show-o"], check=True)
subprocess.run(["git", "-C", "/content/Show-o", "checkout", commit], check=True)


In [ ]:
# Keep Colab's matched PyTorch and torchvision installation.
get_ipython().run_line_magic("pip", 'install transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1')


### Load the model

Select **Runtime > Restart session**, then continue here. Run model loading once;
reuse `adapter` for the following experiments.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch
from IPython.display import display

PROJECT = Path("/content/show-o2-space")
UPSTREAM = Path("/content/Show-o")
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM / "show-o2")]

from show_o2_space.adapter import MODEL_ID, ShowO2Adapter

adapter = ShowO2Adapter()
print(torch.cuda.get_device_name(0), adapter.dtype)


## 2. Generation defaults and baseline

Edit `default_kwargs` to control Show-o2 generation. `model` is our loaded adapter;
`steps` controls flow sampling, `guidance` controls CFG, and `seed` fixes the initial noise.
Run this cell again when changing defaults, then regenerate the baseline.


In [ ]:
model = adapter
default_kwargs = dict(model=model, steps=50, guidance=5.0, seed=42)

def generate_image(prompt, model, steps, guidance, seed):
    return model.generate(prompt, steps=steps, guidance=guidance, seed=seed)

prompt = "A photograph of a car parked on a street."
output = PROJECT / "outputs/steering-red"
output.mkdir(parents=True, exist_ok=True)


In [ ]:
baseline = generate_image(prompt, **default_kwargs)
baseline.save(output / "baseline.png")
metadata = dict(
    model=MODEL_ID, prompt=prompt,
    **{key: default_kwargs[key] for key in ("steps", "guidance", "seed")},
    dtype=str(model.dtype), stage="baseline_only",
    upstream_revision=subprocess.check_output(
        ["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"], text=True
    ).strip(),
    project_revision=subprocess.check_output(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
    ).strip(),
)
(output / "baseline.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
display(baseline)


## 3. Extract and subtract text hidden states

Use the **Show-o2 checkpoint's own backbone**, not a separately loaded Qwen model.
Each text gets a BOS token and a causal attention mask. No image or answer decoding.

`layer` is a zero-based decoder block index. We capture its **output before final
backbone normalization**, matching the project's existing Capture helper. This is
a block-output adaptation; original ActAdd uses block-input activations.
`position=-1` selects the final text token. The pair below ends in red / blue;
inspect the printed tokenization when changing it. No padding or sequence averaging.


In [ ]:
from show_o2_space.capture import Capture

@torch.no_grad()
def extract_text_hidden(text, model, layer):
    token_ids = [model.token_ids["bos_id"]] + model.tokenizer(
        text, add_special_tokens=False
    )["input_ids"]
    input_ids = torch.tensor([token_ids], device=model.device)
    length = input_ids.shape[1]
    # The upstream Qwen forward passes this mask directly to attention.
    causal_mask = torch.full(
        (length, length), float("-inf"), device=model.device, dtype=model.dtype
    ).triu(diagonal=1)[None, None]

    with Capture(model.layers, [layer]) as captured, model.autocast():
        model.model.showo.model(
            input_ids=input_ids, attention_mask=causal_mask,
            use_cache=False, output_hidden_states=False, return_dict=True,
        )
    return captured.states[layer], token_ids


In [ ]:
steering_kwargs = dict(
    layer=8,
    position=-1,
    c=1.0,
    positive_text="The color is red",
    negative_text="The color is blue",
)
print("Available layers:", list(range(len(model.layers))))


In [ ]:
# Rerun this cell after changing layer or either text.
layer = steering_kwargs["layer"]
hidden_red, red_ids = extract_text_hidden(
    steering_kwargs["positive_text"], model=model, layer=layer
)
hidden_control, control_ids = extract_text_hidden(
    steering_kwargs["negative_text"], model=model, layer=layer
)
print("red states:", tuple(hidden_red.shape))
print("control states:", tuple(hidden_control.shape))
print("red tokens:", model.tokenizer.convert_ids_to_tokens(red_ids))
print("control tokens:", model.tokenizer.convert_ids_to_tokens(control_ids))
extraction_metadata = dict(
    layer=layer, positive_text=steering_kwargs["positive_text"],
    negative_text=steering_kwargs["negative_text"],
    red_token_ids=red_ids, control_token_ids=control_ids,
    capture="decoder block output, before final backbone norm",
)


In [ ]:
# Rerun after changing position or c. This does not regenerate images yet.
position = steering_kwargs["position"]
c = steering_kwargs["c"]  # Try 1, 5, 10, 50, or a negative value.

h_red = hidden_red[:, position, :].float()
h_control = hidden_control[:, position, :].float()
v_red = h_red - h_control
steering_vector = c * v_red

print("Selected tokens:",
      model.tokenizer.convert_ids_to_tokens(red_ids[position]),
      model.tokenizer.convert_ids_to_tokens(control_ids[position]))
print("h_red / h_control / v_red:", h_red.shape, h_control.shape, v_red.shape)
print("Norms:", dict(red=h_red.norm().item(), control=h_control.norm().item(),
                     difference=v_red.norm().item(), scaled=steering_vector.norm().item()))


`v_red` is the raw **red-minus-blue contrast**, not an isolated red feature.
`steering_vector` is already multiplied by `c`; inject it once without scaling again.
There is no normalization here: c=50 means fifty times the raw difference, not a
fixed visual effect. Changing `c` only changes this tensor until injection is implemented.


In [ ]:
torch.save(
    dict(h_red=h_red, h_control=h_control, v_red=v_red,
         steering_vector=steering_vector,
         metadata=dict(**extraction_metadata, position=position, c=c, model=MODEL_ID)),
    output / "text_direction.pt",
)
print("Saved:", output / "text_direction.pt")


## 4. Inject during image generation (implementation pending)

Add `steering_vector` (already scaled by c) to selected image-position hidden states at the extraction layer.
Derive the actual image span from the generation sequence layout; exclude text,
padding, time, and delimiter positions. This is a multimodal adaptation of ActAdd.

Make the policy explicit: conditional vs. unconditional CFG rows, and which velocity
evaluations receive the intervention. Show-o2 uses flow matching, so generation
requires multiple backbone calls. Always remove hooks after each run.

First verify c=0 reproduces baseline under identical settings. Then compare
positive and negative coefficients and a matched-norm random-vector control.


## 5. Compare and evaluate (implementation pending)

Display baseline and steered images side by side with identical prompts and seeds.
Report red pixel coverage as a simple proxy, alongside object preservation and
image quality: an overall red tint alone is not evidence of useful attribute control.
Choose layer/intensity on validation prompts, then evaluate held-out prompts and
multiple seeds. Save vectors, positions, layer, coefficients, sampling settings,
and model revision with the results. Pin checkpoint revisions before the study.
